In [ ]:
from selenium.webdriver.support.wait import WebDriverWait
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
def crawl_data(browser, url):
    browser.get(url)

    results = []

    try:
        WebDriverWait(browser, 15).until(
            EC.presence_of_element_located((By.CLASS_NAME, "content-item"))
        )
    except:
        print(f"Không tìm thấy location")
        browser.quit()
        return results
    
    while True:
        try:
            load_more = WebDriverWait(browser, 5).until(
                EC.element_to_be_clickable(
                    (By.XPATH, '//a[@class="fd-btn-more"]/label[contains(text(),"Xem thêm")]'))
            )
            browser.execute_script("arguments[0].click();", load_more)
            WebDriverWait(browser, 10).until(  
                lambda driver: len(driver.find_elements(By.CLASS_NAME, "content-item")) > len(results)
            )
        except Exception:
            break

    stores = browser.find_elements(By.CLASS_NAME, "content-item")

    for store in stores:
        try:
            pref_url = store.find_element(By.CSS_SELECTOR, ".items-content .title a").get_attribute("href")
        except:
            pref_url = ""
        results.append(pref_url)
    browser.close()

    return results 

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--start-maximized")  
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

data = crawl_data(driver, "https://www.foody.vn/ho-chi-minh")
print(len(data))  

In [ ]:
with open("urls.py", "w", encoding="utf-8") as f:
    f.write("urls = [\n")
    for url in data:
        f.write(f"    '{url}',\n")
    f.write("]\n")